In [1]:
import os, glob, unicodedata
import pandas as pd
import numpy as np

d = '../datos/raw/'

def limpiar_nombre(nombre):
    if pd.isna(nombre):
        return ""
    s = ''.join(c for c in unicodedata.normalize('NFD', str(nombre)) if unicodedata.category(c) != 'Mn')
    return s.strip().lower()

def cargar_fbref(tipo):
    lista = []
    for f in sorted(glob.glob(f'{d}fbref_{tipo}_*.csv')):
        s = os.path.basename(f).replace(f'fbref_{tipo}_', '').replace('.csv', '')
        x = pd.read_csv(f, header=1, on_bad_lines='skip')
        x = x[x['Player'].notna() & (x['Player'] != 'Player')].copy()
        x['Season'] = s
        lista.append(x)
    return pd.concat(lista, ignore_index=True)

In [2]:
fb = cargar_fbref('std')
for c in ['Min', 'Gls', 'Ast']:
    fb[c] = pd.to_numeric(fb[c].astype(str).str.replace(',', ''), errors='coerce')
fb = fb.dropna(subset=['Min'])
fb['Player_Clean'] = fb['Player'].apply(limpiar_nombre)

# En tus archivos Gls y Ast vienen por 90: reconstruimos los totales
fb['G_tot'] = (fb['Gls'] * fb['Min'] / 90).round()
fb['A_tot'] = (fb['Ast'] * fb['Min'] / 90).round()
print(len(fb), 'filas FBref;', fb['Season'].nunique(), 'temporadas')

3352 filas FBref; 6 temporadas


In [3]:
lista = []
for f in sorted(glob.glob(d + 'under_2*.csv')):
    s = os.path.basename(f).replace('under_', '').replace('.csv', '')
    x = pd.read_csv(f, sep=None, engine='python')
    x['Season'] = s
    lista.append(x)
us = pd.concat(lista, ignore_index=True)
us['Player_Clean'] = us['player'].apply(limpiar_nombre)

print(us.columns.tolist())
print('Duplicados jugador-temporada en Understat:', us.duplicated(['Player_Clean', 'Season']).sum())

['\ufeff"number"', 'player', 'team', 'apps', 'min', 'goals', 'NPG', 'a', 'xG', 'NPxG', 'xA', 'xGChain', 'xGBuildup', 'xG90', 'NPxG90', 'xA90', 'xG90xA90', 'NPxG90xA90', 'xGChain90', 'xGBuildup90', 'Season', 'Player_Clean']
Duplicados jugador-temporada en Understat: 3


In [4]:
es_fw = fb['Pos'].str.startswith('FW', na=False)
fw_keys = set(zip(fb.loc[es_fw, 'Player_Clean'], fb.loc[es_fw, 'Born']))

agg = (fb.groupby(['Player_Clean', 'Born', 'Season'], as_index=False, dropna=False)
         .agg(Player=('Player', 'first'),
              Squad=('Squad', lambda x: '/'.join(x)),
              Min=('Min', 'sum'),
              G_tot=('G_tot', 'sum'),
              A_tot=('A_tot', 'sum')))

agg['es_fw'] = [(p, b) in fw_keys for p, b in zip(agg['Player_Clean'], agg['Born'])]
agg = agg[agg['es_fw'] & (agg['Min'] >= 900)].copy()
print('Registros jugador-temporada (delanteros, >=900 min):', len(agg))

Registros jugador-temporada (delanteros, >=900 min): 432


In [10]:
equivalencias = {
    'emi buendia': 'emiliano buendia',
    'francisco trincao': 'trincao',
    'hwang hee-chan': 'hee-chan hwang',
    'igor thiago': 'thiago',
    'josh sargent': 'joshua sargent',
}

agg['Player_Clean_fbref'] = agg['Player_Clean']
agg['Player_Clean'] = agg['Player_Clean'].replace(equivalencias)

cruce = agg.merge(us, on=['Player_Clean', 'Season'], how='left', indicator=True)
print(cruce['_merge'].value_counts())
print('Filas en agg:', len(agg), '| filas tras el cruce:', len(cruce))

sin_cruce = cruce[cruce['_merge'] == 'left_only']
print(sin_cruce[['Player', 'Squad', 'Season', 'Min']].to_string())

_merge
both          432
left_only       0
right_only      0
Name: count, dtype: int64
Filas en agg: 432 | filas tras el cruce: 432
Empty DataFrame
Columns: [Player, Squad, Season, Min]
Index: []


In [ ]:
ok = cruce.copy()
ok['ratio_min'] = ok['min'] / ok['Min']
ok['dif_goles'] = (ok['goals'] - ok['G_tot']).abs()

sospechosos = ok[(ok['ratio_min'] < 0.85) | (ok['ratio_min'] > 1.15) | (ok['dif_goles'] > 3)]
print(len(sospechosos), 'registros a revisar de', len(ok))
print(sospechosos[['Player', 'Squad', 'team', 'Season', 'Min', 'min', 'G_tot', 'goals']].to_string())

In [ ]:
cols = ['Player', 'Player_Clean', 'Born', 'Season', 'Squad', 'Min', 'G_tot', 'A_tot',
        'team', 'min', 'goals', 'NPG', 'a', 'xG', 'NPxG', 'xA', 'xG90', 'NPxG90', 'xA90']
cols = [c for c in cols if c in cruce.columns]

base = cruce[cols].copy()
os.makedirs('../datos/procesado', exist_ok=True)
base.to_csv('../datos/procesado/fw_base.csv', index=False)
print(base.shape)
display(base.head())
sht = cargar_fbref('sht')
print(sht.columns.tolist())
print(sht[sht['Player'] == 'Mohamed Salah'].head(3).to_string())

In [16]:
sht = cargar_fbref('sht')
for c in ['90s', 'Sh', 'SoT']:
    sht[c] = pd.to_numeric(sht[c], errors='coerce')
sht['Player_Clean'] = sht['Player'].apply(limpiar_nombre)

# Totales reconstruidos desde las tasas por 90
sht['Sh_tot'] = (sht['Sh'] * sht['90s']).round()
sht['SoT_tot'] = (sht['SoT'] * sht['90s']).round()

sht['Player_Clean'] = sht['Player_Clean'].replace(equivalencias)

sht_agg = (sht.groupby(['Player_Clean', 'Born', 'Season'], as_index=False, dropna=False)
              .agg(Sh_tot=('Sh_tot', 'sum'), SoT_tot=('SoT_tot', 'sum')))

In [ ]:
base = cruce.merge(sht_agg, on=['Player_Clean', 'Born', 'Season'],
                   how='left', indicator='_m_sht')
print(base['_m_sht'].value_counts())
print('Filas:', len(cruce), '->', len(base))
print(base[['Sh_tot', 'SoT_tot']].isna().sum())
print('SoT > Sh (debe ser 0):', (base['SoT_tot'] > base['Sh_tot']).sum())

# Todo por 90 con una sola base de minutos (FBref)
n90 = base['Min'] / 90
base['Sh_p90'] = base['Sh_tot'] / n90
base['SoT_p90'] = base['SoT_tot'] / n90
base['npG_p90'] = base['NPG'] / n90
base['NPxG_p90'] = base['NPxG'] / n90
base['xA_p90'] = base['xA'] / n90
base['npG_menos_NPxG_p90'] = (base['NPG'] - base['NPxG']) / n90

crit = ['Sh_p90', 'SoT_p90', 'npG_p90', 'NPxG_p90', 'xA_p90', 'npG_menos_NPxG_p90']
print(base[crit].describe().round(2))

# Control con Salah
print(base.loc[base['Player_Clean'] == 'mohamed salah',
               ['Season', 'Min', 'Sh_tot', 'SoT_tot', 'npG_p90', 'NPxG_p90']].round(2).to_string())

In [20]:
tm = '../datos/raw/transfermarkt/'
pl = pd.read_csv(tm + 'players.csv',
                 usecols=['player_id', 'name', 'date_of_birth', 'position', 'sub_position'])
pl['Born'] = pd.to_datetime(pl['date_of_birth'], errors='coerce').dt.year
pl['key'] = pl['name'].apply(limpiar_nombre)

# Nombre original de FBref (no el de Understat)
base['key'] = base['Player'].apply(limpiar_nombre)

ident = base[['key', 'Born']].drop_duplicates()
m = ident.merge(pl[['key', 'Born', 'player_id', 'name', 'position']],
                on=['key', 'Born'], how='left', indicator=True)

print(m['_merge'].value_counts())
print('Jugadores con más de un player_id:', m.duplicated(['key', 'Born']).sum())
print(m['position'].value_counts(dropna=False))
print(m[m['_merge'] == 'left_only'][['key', 'Born']].to_string())

_merge
both          168
left_only       6
right_only      0
Name: count, dtype: int64
Jugadores con más de un player_id: 0
position
Attack      156
Midfield     12
NaN           6
Name: count, dtype: int64
                      key    Born
48            emi buendia  1996.0
67         hwang hee-chan  1996.0
117       mykhailo mudryk  2001.0
126       oliver mcburnie  1996.0
153         son heung-min  1992.0
163  valentin castellanos  1998.0


In [23]:
tm = '../datos/raw/transfermarkt/'

equivalencias_tm = {
    'emi buendia': 'emiliano buendia',
    'hwang hee-chan': 'hee-chan hwang',
    'mykhailo mudryk': 'mykhaylo mudryk',
    'oliver mcburnie': 'oli mcburnie',
    'son heung-min': 'heung-min son',
    'valentin castellanos': 'taty castellanos',
}

pl = pd.read_csv(tm + 'players.csv',
                 usecols=['player_id', 'name', 'date_of_birth', 'position', 'sub_position'])
pl['Born'] = pd.to_datetime(pl['date_of_birth'], errors='coerce').dt.year
pl['key'] = pl['name'].apply(limpiar_nombre)

base['key'] = base['Player'].apply(limpiar_nombre).replace(equivalencias_tm)

ident = base[['key', 'Born']].drop_duplicates()
m = ident.merge(pl[['key', 'Born', 'player_id', 'name', 'position']],
                on=['key', 'Born'], how='left', indicator=True)

print(m['_merge'].value_counts())
print('Jugadores con más de un player_id:', m.duplicated(['key', 'Born']).sum())
print(m['position'].value_counts(dropna=False))
print(m[m['_merge'] == 'left_only'][['key', 'Born']].to_string())

_merge
both          174
left_only       0
right_only      0
Name: count, dtype: int64
Jugadores con más de un player_id: 0
position
Attack      162
Midfield     12
Name: count, dtype: int64
Empty DataFrame
Columns: [key, Born]
Index: []


In [25]:
val = pd.read_csv(tm + 'player_valuations.csv',
                  usecols=['player_id', 'date', 'market_value_in_eur', 'current_club_name'])
val['date'] = pd.to_datetime(val['date']).astype('datetime64[ns]')
val['player_id'] = val['player_id'].astype('int64')
val = val.rename(columns={'current_club_name': 'club_tm'}).sort_values('date')

b = base.merge(m[['key', 'Born', 'player_id']], on=['key', 'Born'], how='left')
b['fecha_corte'] = pd.to_datetime('20' + b['Season'].str[-2:] + '-06-01').astype('datetime64[ns]')

con_id = b.dropna(subset=['player_id']).copy()
con_id['player_id'] = con_id['player_id'].astype('int64')
con_id = con_id.sort_values('fecha_corte')

costo = pd.merge_asof(con_id, val, left_on='fecha_corte', right_on='date',
                      by='player_id', direction='backward',
                      tolerance=pd.Timedelta(days=365))
costo['dias_desde_valoracion'] = (costo['fecha_corte'] - costo['date']).dt.days

print('Filas:', len(base), '->', len(costo))
print('Sin valoración:', costo['market_value_in_eur'].isna().sum())
print(costo['dias_desde_valoracion'].describe().round(0))
print(costo.loc[costo['key'] == 'mohamed salah',
                ['Season', 'Squad', 'club_tm', 'date', 'market_value_in_eur']].to_string())

Filas: 432 -> 432
Sin valoración: 0
count    432.0
mean      83.0
std       73.0
min        0.0
25%        5.0
50%       75.0
75%      160.0
max      210.0
Name: dias_desde_valoracion, dtype: float64
    Season      Squad       club_tm       date  market_value_in_eur
19   20-21  Liverpool  Liverpool FC 2021-03-18            110000000
107  21-22  Liverpool  Liverpool FC 2021-12-23            100000000
171  22-23  Liverpool  Liverpool FC 2023-03-16             70000000
247  23-24  Liverpool  Liverpool FC 2024-05-27             55000000
350  24-25  Liverpool  Liverpool FC 2025-05-30             50000000
430  25-26  Liverpool  Liverpool FC 2025-12-09             30000000


In [26]:
import numpy as np

# 1. Aislar el mercado actual (ej. temporada 25-26) para simular contrataciones reales
df_actual = costo[costo['Season'] == '25-26'].reset_index(drop=True)

# 2. Definir la matriz multicriterio
# Beneficios: Goles esperados sin penal, Tiros al arco, Asistencias esperadas
# Costo: Valor de mercado en Euros
criterios = ['NPxG_p90', 'SoT_p90', 'xA_p90', 'market_value_in_eur']

# Extraer la matriz de decisión (X)
X = df_actual[criterios].values

# 3. Parámetros del modelo
# Pesos: 40% Calidad de tiro, 15% Volumen, 15% Creación, 30% Costo económico
# (En un proyecto más avanzado, podrías usar AHP para calcular estos pesos exactos)
pesos = np.array([0.40, 0.15, 0.15, 0.30]) 
impactos = np.array([1, 1, 1, -1]) # 1 = Maximizar (Beneficio), -1 = Minimizar (Costo)

# 4. Algoritmo TOPSIS Vectorial
# Normalización de la matriz
norm_X = X / np.sqrt((X**2).sum(axis=0))

# Matriz ponderada
V = norm_X * pesos

# Determinar las soluciones ideales (Mejor y Peor)
ideal_best = np.where(impactos == 1, V.max(axis=0), V.min(axis=0))
ideal_worst = np.where(impactos == 1, V.min(axis=0), V.max(axis=0))

# Calcular distancias euclidianas
dist_best = np.sqrt(((V - ideal_best)**2).sum(axis=1))
dist_worst = np.sqrt(((V - ideal_worst)**2).sum(axis=1))

# Calcular el puntaje final (Coeficiente de cercanía)
df_actual['Puntaje_TOPSIS'] = dist_worst / (dist_best + dist_worst)

# 5. Resultados: Los delanteros más eficientes del mercado
resultados_moneyball = df_actual.sort_values('Puntaje_TOPSIS', ascending=False)

print("🏆 Top 10 Delanteros Infravalorados (Alta Producción / Bajo Costo)")
columnas_vista = ['key', 'Squad', 'NPxG_p90', 'market_value_in_eur', 'Puntaje_TOPSIS']
display(resultados_moneyball[columnas_vista].head(10).round({'NPxG_p90': 2, 'Puntaje_TOPSIS': 4}))

🏆 Top 10 Delanteros Infravalorados (Alta Producción / Bajo Costo)


,key,Squad,NPxG_p90,market_value_in_eur,Puntaje_TOPSIS
13,beto,Everton,0.63,20000000,0.7633
52,callum wilson,West Ham,0.55,5000000,0.7629
15,lukas nmecha,Leeds United,0.49,10000000,0.7562
39,ollie watkins,Aston Villa,0.60,30000000,0.7537
54,richarlison,Tottenham,0.45,28000000,0.7141
55,danny welbeck,Brighton,0.42,4000000,0.7141
43,dominic calvert-lewin,Leeds United,0.49,22000000,0.7128
48,jean-philippe mateta,Crystal Palace,0.52,35000000,0.7057
11,taty castellanos,West Ham,0.46,28000000,0.6964
62,zian flemming,Burnley,0.41,12000000,0.6881
